In [1]:
import pandas as pd
import numpy as np

df_feat = pd.read_pickle("dataset_features_v3.pkl")

In [106]:
sample_rank = df_feat.iloc[:1000].copy()

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
skill_embeddings = model.encode(
    sample_rank['skills_text'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

career_embeddings = model.encode(
    sample_rank['career_text'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

profile_embeddings = model.encode(
    sample_rank['profile_text'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [55]:
jd_skill_text = """
embeddings retrieval ranking recommendation systems
hybrid search vector databases
sentence-transformers BGE E5 FAISS Pinecone Weaviate Qdrant OpenSearch Elasticsearch
Python evaluation frameworks NDCG MAP MRR
"""

jd_career_text = """
product company experience
production ML systems
retrieval ranking recommendation search systems
evaluation frameworks
A/B testing
NDCG MAP MRR
Python engineering
hands-on coding
shipper mindset
"""

jd_profile_text = """
senior AI engineer
6 to 8 years experience
active candidate
open to relocation
startup mindset
fast moving engineering culture
hands-on coding
scalable systems
"""

In [107]:
jd_skill_embedding = model.encode(
    jd_skill_text,
    normalize_embeddings=True
)

jd_career_embedding = model.encode(
    jd_career_text,
    normalize_embeddings=True
)

jd_profile_embedding = model.encode(
    jd_profile_text,
    normalize_embeddings=True
)

In [108]:
skill_similarity = cosine_similarity(
    skill_embeddings,
    jd_skill_embedding.reshape(1,-1)
).flatten()

career_similarity = cosine_similarity(
    career_embeddings,
    jd_career_embedding.reshape(1,-1)
).flatten()

profile_similarity = cosine_similarity(
    profile_embeddings,
    jd_profile_embedding.reshape(1,-1)
).flatten()

In [111]:
sample_rank['semantic_score'] = (
    0.70*career_similarity
    +0.25*skill_similarity
    +0.05*profile_similarity
)

In [112]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
sample_rank['semantic_score_norm'] = scaler.fit_transform(
    sample_rank[['semantic_score']]
)

In [113]:
career_cols = [
    'profile_years_of_experience',
    'current_company_tenure_months',
    'avg_job_duration_months',
    'highest_degree_score'
]

sample_rank['career_score'] = scaler.fit_transform(
    sample_rank[career_cols]
    .mean(axis=1)
    .values.reshape(-1,1)
)

In [114]:
signal_cols = [c for c in sample_rank.columns if 'signal_' in c]
print(signal_cols)

['signal_profile_completeness_score', 'signal_signup_date', 'signal_last_active_date', 'signal_open_to_work_flag', 'signal_profile_views_received_30d', 'signal_applications_submitted_30d', 'signal_recruiter_response_rate', 'signal_avg_response_time_hours', 'signal_connection_count', 'signal_endorsements_received', 'signal_notice_period_days', 'signal_preferred_work_mode', 'signal_willing_to_relocate', 'signal_github_activity_score', 'signal_search_appearance_30d', 'signal_saved_by_recruiters_30d', 'signal_interview_completion_rate', 'signal_offer_acceptance_rate', 'signal_verified_email', 'signal_verified_phone', 'signal_linkedin_connected', 'signal_skill_assessment_scores.NLP', 'signal_skill_assessment_scores.Image Classification', 'signal_skill_assessment_scores.Fine-tuning LLMs', 'signal_skill_assessment_scores.Speech Recognition', 'signal_expected_salary_range_inr_lpa.min', 'signal_expected_salary_range_inr_lpa.max', 'signal_skill_assessment_scores.GANs', 'signal_skill_assessment_s

In [115]:
behavior_cols = [
    'signal_recruiter_response_rate',
    'signal_interview_completion_rate',
    'signal_github_activity_score',
    'signal_saved_by_recruiters_30d',
    'signal_profile_views_received_30d'
]

In [116]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
sample_rank[behavior_cols] = scaler.fit_transform(
    sample_rank[behavior_cols]
)

sample_rank['behavior_score'] = (
    sample_rank[behavior_cols]
    .mean(axis=1)
)

In [118]:
availability_cols = [
    'signal_open_to_work_flag',
    'signal_willing_to_relocate'
]
sample_rank['signal_last_active_date'] = pd.to_datetime(
    sample_rank['signal_last_active_date']
)

today = pd.Timestamp.today()

sample_rank['days_since_last_active'] = (
    today - sample_rank['signal_last_active_date']
).dt.days

In [119]:
sample_rank['recent_activity_score'] = 1 - MinMaxScaler().fit_transform(
    sample_rank[['days_since_last_active']]
)

In [120]:
sample_rank['availability_score'] = (
    0.4*sample_rank['signal_open_to_work_flag']
    +0.2*sample_rank['signal_willing_to_relocate']
    +0.4*sample_rank['recent_activity_score']
)

In [121]:
skill_cols = [
    'num_skills',
    'avg_skill_duration_months',
    'avg_skill_endorsements'
]
sample_rank[skill_cols] = scaler.fit_transform(
    sample_rank[skill_cols]
)

sample_rank['skill_score'] = (
    sample_rank[skill_cols]
    .mean(axis=1)
)

In [122]:
sorted(sample_rank['profile_current_title'].dropna().unique())

['.NET Developer',
 'AI Research Engineer',
 'AI Specialist',
 'Accountant',
 'Analytics Engineer',
 'Backend Engineer',
 'Business Analyst',
 'Civil Engineer',
 'Cloud Engineer',
 'Content Writer',
 'Customer Support',
 'Data Analyst',
 'Data Engineer',
 'Data Scientist',
 'DevOps Engineer',
 'Frontend Engineer',
 'Full Stack Developer',
 'Graphic Designer',
 'HR Manager',
 'Java Developer',
 'ML Engineer',
 'Marketing Manager',
 'Mechanical Engineer',
 'Mobile Developer',
 'Operations Manager',
 'Project Manager',
 'QA Engineer',
 'Recommendation Systems Engineer',
 'Sales Executive',
 'Senior Data Engineer',
 'Senior Software Engineer',
 'Software Engineer']

In [123]:
strong_positive = [
    'Recommendation Systems Engineer',
    'ML Engineer',
    'AI Research Engineer',
    'AI Specialist',
    'Backend Engineer',
    'Software Engineer',
    'Senior Software Engineer',
    'Data Scientist',
    'Data Engineer',
    'Senior Data Engineer',
    'Analytics Engineer',
    'Cloud Engineer',
    'DevOps Engineer'
]
mild_positive = [
    '.NET Developer',
    'Java Developer',
    'Frontend Engineer',
    'Full Stack Developer',
    'Mobile Developer'
]
neutral = [
    'Business Analyst',
    'QA Engineer',
    'Project Manager',
    'Data Analyst'
]
strong_negative = [
    'HR Manager',
    'Marketing Manager',
    'Content Writer',
    'Graphic Designer',
    'Customer Support',
    'Sales Executive',
    'Accountant'
]
mild_negative = [
    'Mechanical Engineer',
    'Civil Engineer',
    'Operations Manager'
]

In [124]:
def title_prior(title):
    
    if title in strong_positive:
        return 1

    elif title in mild_positive:
        return 0.5

    elif title in strong_negative:
        return -1

    elif title in mild_negative:
        return -0.5

    else:
        return 0

In [125]:
sample_rank['title_prior'] = (
    sample_rank['profile_current_title']
    .apply(title_prior)
)

In [126]:
non_ai_titles = [
    'HR Manager',
    'Marketing Manager',
    'Content Writer',
    'Graphic Designer',
    'Customer Support',
    'Sales Executive',
    'Accountant'
]

ai_words = [
    'rag',
    'langchain',
    'pinecone',
    'embeddings',
    'vector search',
    'genai',
    'llm',
    'openai'
]

In [127]:
def transition_penalty(row):

    title = str(row['profile_current_title']).lower()
    profile = str(row['profile_text']).lower()

    if (
        any(t.lower() == title for t in non_ai_titles)
        and
        any(word in profile for word in ai_words)
    ):
        return -1

    return 0

In [128]:
sample_rank['transition_penalty'] = (
    sample_rank.apply(
        transition_penalty,
        axis=1
    )
)

In [129]:
sample_rank['job_switch_penalty'] = (
    sample_rank['job_switch_count'] > 8
).astype(int)
sample_rank['inactive_penalty'] = (
    sample_rank['days_since_last_active'] > 180
).astype(int)

In [130]:
sample_rank['integrity_penalty'] = (
    sample_rank['job_switch_penalty']
    +
    sample_rank['inactive_penalty']
)
from sklearn.preprocessing import MinMaxScaler

sample_rank['integrity_penalty'] = (
    MinMaxScaler()
    .fit_transform(
        sample_rank[['integrity_penalty']]
    )
)

In [131]:
sample_rank['final_score'] = (
      0.40*sample_rank['semantic_score_norm']
    + 0.20*sample_rank['career_score']
    + 0.15*sample_rank['behavior_score']
    + 0.10*sample_rank['availability_score']
    + 0.10*sample_rank['skill_score']
    - 0.05*sample_rank['integrity_penalty']
    + 0.10*sample_rank['title_prior']
    + 0.10*sample_rank['transition_penalty']
)

In [132]:
sample_rank[
[
'candidate_id',
'profile_current_title',
'final_score'
]
].sort_values(
'final_score',
ascending=False
).head(20)

,candidate_id,profile_current_title,final_score
30,CAND_0000031,Recommendation Systems Engineer,0.811300
272,CAND_0000273,ML Engineer,0.792584
980,CAND_0000981,ML Engineer,0.755980
164,CAND_0000165,AI Specialist,0.734778
199,CAND_0000200,ML Engineer,0.684965
421,CAND_0000422,AI Research Engineer,0.684520
583,CAND_0000584,Analytics Engineer,0.680047
61,CAND_0000062,DevOps Engineer,0.672158
9,CAND_0000010,Data Engineer,0.671109
111,CAND_0000112,AI Specialist,0.669753


In [133]:
cols = [
'profile_current_title',
'profile_years_of_experience',
'skills_text',
'career_text',
'signal_recruiter_response_rate',
'signal_github_activity_score',
'signal_notice_period_days'
]

sample_rank.sort_values(
    'final_score',
    ascending=False
)[cols].head(10)

,profile_current_title,profile_years_of_experience,skills_text,career_text,signal_recruiter_response_rate,signal_github_activity_score,signal_notice_period_days
30,Recommendation Systems Engineer,6.0,Go MLflow FAISS Pinecone Angular Image Classif...,Recommendation Systems Engineer Food Delivery ...,1.000000,0.417910,60
272,ML Engineer,5.8,TTS Forecasting Spring Boot Fine-tuning LLMs F...,ML Engineer EdTech Built recommendation-style ...,0.494253,0.375622,60
980,ML Engineer,6.4,Semantic Search Computer Vision Image Classifi...,ML Engineer HealthTech AI Built recommendation...,0.586207,0.886816,120
164,AI Specialist,6.8,BentoML Information Retrieval Semantic Search ...,AI Specialist AI/ML Built computer vision mode...,0.298851,0.777363,120
199,ML Engineer,4.3,BigQuery Vector Search Prompt Engineering Bent...,ML Engineer AI/ML Worked on customer-facing pr...,0.643678,0.000000,90
421,AI Research Engineer,6.3,MLflow Python Photoshop Milvus Java OpenCV Lea...,AI Research Engineer Conversational AI Built N...,0.862069,0.000000,90
583,Analytics Engineer,7.3,BM25 Semantic Search ASR CI/CD JavaScript Node...,Analytics Engineer Manufacturing Backend + dat...,0.873563,0.000000,90
61,DevOps Engineer,4.3,Angular Hadoop GANs Kubernetes MLOps TypeScrip...,DevOps Engineer IT Services Test automation an...,0.747126,0.583333,90
9,Data Engineer,4.6,GCP Spring Boot Kubeflow Java GANs Figma Elast...,Data Engineer Transportation Mixed data scienc...,0.413793,0.431592,120
111,AI Specialist,3.8,Deep Learning TensorFlow Time Series Speech Re...,AI Specialist Food Delivery Built NLP pipeline...,0.310345,1.000000,45


In [135]:

sample_rank.loc[
    sample_rank['signal_notice_period_days'] > 90,
    'notice_penalty'
] = 1.0

sample_rank.loc[
    (sample_rank['signal_notice_period_days'] > 30) &
    (sample_rank['signal_notice_period_days'] <= 90),
    'notice_penalty'
] = 0.5

In [136]:
keywords = [
    'recommendation',
    'retrieval',
    'ranking',
    'semantic search',
    'information retrieval',
    'vector search',
    'faiss',
    'bm25'
]
def retrieval_bonus(row):
    text = (
        str(row['skills_text']) +
        " " +
        str(row['career_text'])
    ).lower()

    return int(
        any(word in text for word in keywords)
    )

In [137]:
sample_rank['retrieval_bonus'] = (
    sample_rank.apply(
        retrieval_bonus,
        axis=1
    )
)

In [138]:
sample_rank['profile_current_industry'].value_counts()

profile_current_industry
IT Services          295
Manufacturing        226
Software             214
Paper Products        73
Conglomerate          71
Food Delivery         35
Fintech               33
Consulting            19
E-commerce            12
EdTech                 9
SaaS                   3
AI/ML                  2
HealthTech             2
Gaming                 2
Transportation         1
Conversational AI      1
AdTech                 1
HealthTech AI          1
Name: count, dtype: int64

In [139]:
product_industries = [
    'Software',
    'Food Delivery',
    'Fintech',
    'E-commerce',
    'EdTech',
    'SaaS',
    'AI/ML',
    'HealthTech',
    'Gaming',
    'Conversational AI',
    'AdTech',
    'HealthTech AI'
]
negative_industries = [
    'IT Services',
    'Manufacturing',
    'Consulting'
]

In [140]:
def industry_bonus(industry):
    if industry in product_industries:
        return 1
    elif industry in negative_industries:
        return -1
    return 0
sample_rank['industry_bonus'] = (
    sample_rank['profile_current_industry']
    .apply(industry_bonus)
)

In [141]:
sample_rank['final_score'] = (
      0.40*sample_rank['semantic_score_norm']
    + 0.20*sample_rank['career_score']
    + 0.15*sample_rank['behavior_score']
    + 0.10*sample_rank['availability_score']
    + 0.10*sample_rank['skill_score']
    - 0.05*sample_rank['integrity_penalty']
    + 0.10*sample_rank['title_prior']
    + 0.10*sample_rank['transition_penalty']
    + 0.05*sample_rank['retrieval_bonus']
    - 0.05*sample_rank['notice_penalty']
    + 0.05*sample_rank['industry_bonus']
)

In [142]:
sample_rank[
[
'candidate_id',
'profile_current_title',
'final_score'
]
].sort_values(
'final_score',
ascending=False
).head(20)

,candidate_id,profile_current_title,final_score
30,CAND_0000031,Recommendation Systems Engineer,0.886300
272,CAND_0000273,ML Engineer,0.867584
980,CAND_0000981,ML Engineer,0.805980
164,CAND_0000165,AI Specialist,0.784778
199,CAND_0000200,ML Engineer,0.759965
421,CAND_0000422,AI Research Engineer,0.759520
111,CAND_0000112,AI Specialist,0.744753
664,CAND_0000665,Senior Software Engineer,0.733278
858,CAND_0000859,Senior Software Engineer,0.718212
968,CAND_0000969,AI Specialist,0.708203


In [143]:
sample_rank.loc[
    sample_rank['signal_notice_period_days'] > 90,
    'notice_penalty'
] = 2.0

In [145]:
service_companies = [
    'TCS',
    'Infosys',
    'Wipro',
    'Accenture',
    'Cognizant',
    'Capgemini'
]
def career_company_penalty(career_history):

    companies = [
        str(job.get('company', ''))
        for job in career_history
    ]

    if len(companies) == 0:
        return 0

    count_service = sum(
        any(
            s.lower() in company.lower()
            for s in service_companies
        )
        for company in companies
    )

    # Entire career in service companies
    if count_service == len(companies):
        return 1
    return 0

sample_rank['career_company_penalty'] = (
    sample_rank['career_history']
    .apply(career_company_penalty)
)

In [146]:
sample_rank['final_score'] = (
      0.40*sample_rank['semantic_score_norm']
    + 0.20*sample_rank['career_score']
    + 0.15*sample_rank['behavior_score']
    + 0.10*sample_rank['availability_score']
    + 0.10*sample_rank['skill_score']
    - 0.05*sample_rank['integrity_penalty']
    + 0.10*sample_rank['title_prior']
    + 0.10*sample_rank['transition_penalty']
    + 0.05*sample_rank['retrieval_bonus']
    - 0.05*sample_rank['notice_penalty']
    + 0.05*sample_rank['industry_bonus']
    - 0.05*sample_rank['career_company_penalty']
)

In [147]:
sample_rank[
[
'candidate_id',
'profile_current_title',
'final_score'
]
].sort_values(
'final_score',
ascending=False
).head(20)

,candidate_id,profile_current_title,final_score
30,CAND_0000031,Recommendation Systems Engineer,0.886300
272,CAND_0000273,ML Engineer,0.867584
199,CAND_0000200,ML Engineer,0.759965
421,CAND_0000422,AI Research Engineer,0.759520
980,CAND_0000981,ML Engineer,0.755980
111,CAND_0000112,AI Specialist,0.744753
164,CAND_0000165,AI Specialist,0.734778
664,CAND_0000665,Senior Software Engineer,0.733278
858,CAND_0000859,Senior Software Engineer,0.718212
968,CAND_0000969,AI Specialist,0.708203


In [148]:
sample_rank.loc[
    sample_rank['signal_notice_period_days'] > 90,
    'notice_penalty'
] = 2.0
sample_rank.loc[
    (sample_rank['signal_notice_period_days'] > 30)
    &
    (sample_rank['signal_notice_period_days'] <= 90),
    'notice_penalty'
] = 1.0

In [149]:
sample_rank[
[
'candidate_id',
'profile_current_title',
'final_score'
]
].sort_values(
'final_score',
ascending=False
).head(20)

,candidate_id,profile_current_title,final_score
30,CAND_0000031,Recommendation Systems Engineer,0.886300
272,CAND_0000273,ML Engineer,0.867584
199,CAND_0000200,ML Engineer,0.759965
421,CAND_0000422,AI Research Engineer,0.759520
980,CAND_0000981,ML Engineer,0.755980
111,CAND_0000112,AI Specialist,0.744753
164,CAND_0000165,AI Specialist,0.734778
664,CAND_0000665,Senior Software Engineer,0.733278
858,CAND_0000859,Senior Software Engineer,0.718212
968,CAND_0000969,AI Specialist,0.708203


In [150]:
cols = [
'profile_current_title',
'profile_years_of_experience',
'profile_current_industry',
'signal_notice_period_days',
'signal_recruiter_response_rate',
'skills_text',
'career_text'
]

sample_rank.sort_values(
    'final_score',
    ascending=False
)[cols].head(10)

,profile_current_title,profile_years_of_experience,profile_current_industry,signal_notice_period_days,signal_recruiter_response_rate,skills_text,career_text
30,Recommendation Systems Engineer,6.0,Food Delivery,60,1.000000,Go MLflow FAISS Pinecone Angular Image Classif...,Recommendation Systems Engineer Food Delivery ...
272,ML Engineer,5.8,EdTech,60,0.494253,TTS Forecasting Spring Boot Fine-tuning LLMs F...,ML Engineer EdTech Built recommendation-style ...
199,ML Engineer,4.3,AI/ML,90,0.643678,BigQuery Vector Search Prompt Engineering Bent...,ML Engineer AI/ML Worked on customer-facing pr...
421,AI Research Engineer,6.3,Conversational AI,90,0.862069,MLflow Python Photoshop Milvus Java OpenCV Lea...,AI Research Engineer Conversational AI Built N...
980,ML Engineer,6.4,HealthTech AI,120,0.586207,Semantic Search Computer Vision Image Classifi...,ML Engineer HealthTech AI Built recommendation...
111,AI Specialist,3.8,Food Delivery,45,0.310345,Deep Learning TensorFlow Time Series Speech Re...,AI Specialist Food Delivery Built NLP pipeline...
164,AI Specialist,6.8,AI/ML,120,0.298851,BentoML Information Retrieval Semantic Search ...,AI Specialist AI/ML Built computer vision mode...
664,Senior Software Engineer,6.2,HealthTech,90,0.172414,Recommendation Systems CSS Object Detection NL...,Senior Software Engineer HealthTech Built and ...
858,Senior Software Engineer,5.9,EdTech,90,0.839080,Pinecone SEO Docker CNN Semantic Search Spring...,Senior Software Engineer EdTech Backend develo...
968,AI Specialist,5.5,EdTech,60,0.873563,Pinecone BigQuery Vue.js Semantic Search Apach...,AI Specialist EdTech Built computer vision mod...


In [151]:
ir_words = [
    'recommendation',
    'retrieval',
    'ranking',
    'semantic search',
    'information retrieval',
    'vector search',
    'faiss',
    'bm25',
    'sentence transformers',
    'pinecone',
    'qdrant',
    'weaviate',
    'elasticsearch',
    'opensearch'
]


In [152]:
def ir_bonus(row):
    
    text = (
        str(row['career_text']) + " " +
        str(row['skills_text'])
    ).lower()
    
    return int(
        any(word in text for word in ir_words)
    )
sample_rank['ir_bonus'] = (
    sample_rank.apply(
        ir_bonus,
        axis=1
    )
)

In [153]:
sample_rank['final_score'] = (
      0.40*sample_rank['semantic_score_norm']
    + 0.20*sample_rank['career_score']
    + 0.15*sample_rank['behavior_score']
    + 0.10*sample_rank['availability_score']
    + 0.10*sample_rank['skill_score']
    - 0.05*sample_rank['integrity_penalty']
    + 0.10*sample_rank['title_prior']
    + 0.10*sample_rank['transition_penalty']
    + 0.05*sample_rank['retrieval_bonus']
    - 0.05*sample_rank['notice_penalty']
    + 0.05*sample_rank['industry_bonus']
    - 0.05*sample_rank['career_company_penalty']
    + 0.05*sample_rank['ir_bonus']
)

In [154]:
sample_rank[
[
'candidate_id',
'profile_current_title',
'final_score'
]
].sort_values(
'final_score',
ascending=False
).head(20)

,candidate_id,profile_current_title,final_score
30,CAND_0000031,Recommendation Systems Engineer,0.911300
272,CAND_0000273,ML Engineer,0.892584
980,CAND_0000981,ML Engineer,0.805980
199,CAND_0000200,ML Engineer,0.784965
164,CAND_0000165,AI Specialist,0.784778
421,CAND_0000422,AI Research Engineer,0.784520
111,CAND_0000112,AI Specialist,0.769753
664,CAND_0000665,Senior Software Engineer,0.758278
858,CAND_0000859,Senior Software Engineer,0.743212
968,CAND_0000969,AI Specialist,0.733203
